In [1]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import train_test_split

In [ ]:
# Check if model already exists and load it
import os
model_path = 'email_reply_model.pth'
vectorizer_path = 'vectorizer.pth'

if os.path.exists(model_path):
    print("Loading existing trained model...")
    
    # Load dataset to recreate vectorizer if needed
    with open("emails.json", 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    bodies = [email['body'] for email in dataset]
    
    if os.path.exists(vectorizer_path):
        # Load vectorizer if it exists
        vectorizer = torch.load(vectorizer_path)
    else:
        print("Vectorizer not found, recreating from dataset...")
        # Recreate vectorizer from training data
        vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        vectorizer.fit_transform(bodies)
        # Save it for future use
        torch.save(vectorizer, vectorizer_path)
        print("Vectorizer recreated and saved.")
    
    # Determine input_dims from vectorizer
    X_sample = vectorizer.transform(bodies[:1]).toarray()
    input_dims = X_sample.shape[1]
    
    # Create model and load state
    model = EmailReply(input_dims, 2)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    print("Model loaded successfully!")
    
    # Skip to prediction function
    # Define predict_reply function here too
    def predict_reply(email_body):
        """Predict if an email should be replied to."""
        model.eval()
        X_new = vectorizer.transform([email_body]).toarray()
        X_tensor = torch.tensor(X_new, dtype=torch.float32)
        with torch.no_grad():
            output = model(X_tensor)
            _, predicted = torch.max(output, 1)
            return predicted.item() == 1  # True if replied
    
else:
    print("Model not found, will train new model...")

In [2]:
class EmailReply(nn.Module):
    def __init__(self, input_dims, output_dims):
        # Should be a neural network trained on a large corpus of emails and finetuned on my emails
        super(EmailReply, self).__init__()
        self.dims = [512, 256]
        self.layers = nn.Sequential(
            nn.Linear(input_dims, self.dims[0]),
            nn.ReLU(),
            nn.Linear(self.dims[0], self.dims[1]),
            nn.ReLU(),
            nn.Linear(self.dims[1], output_dims)
        )

    def forward(self, input):
        return self.layers(input)

In [3]:
with open("emails.json", 'r', encoding='utf-8') as f:
    dataset = json.load(f)

In [4]:
# Extract bodies and labels
bodies = [email['body'] for email in dataset]
labels = [1 if email['replied'] else 0 for email in dataset]
print(labels.count(1))
print(labels.count(0))
print(len(labels))

2155
23979
26134


In [8]:
# Vectorize text
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(bodies).toarray()
y = np.array(labels)

print(X.shape)

(26134, 5000)


In [21]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

input_dims = X.shape[1]
output_dims = 2  # replied or not

In [22]:
model = EmailReply(input_dims, output_dims)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
num_epochs = 100
batch_size = 64

In [23]:
num_epochs = 10
batch_size = 64

for ep in range(num_epochs):
    # Shuffle data
    indices = torch.randperm(len(x_train))
    x_train_shuffled = x_train[indices]
    y_train_shuffled = y_train[indices]

    epoch_loss = 0
    num_batches = len(x_train) // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        x_batch = x_train_shuffled[start_idx:end_idx]
        y_batch = y_train_shuffled[start_idx:end_idx]

        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / num_batches
    print(f"Epoch {ep+1}: loss {avg_loss:.4f}")

Epoch 1: loss 0.4335
Epoch 2: loss 0.2113
Epoch 3: loss 0.1689
Epoch 4: loss 0.1390
Epoch 5: loss 0.1184
Epoch 6: loss 0.1069
Epoch 7: loss 0.0958
Epoch 8: loss 0.0888
Epoch 9: loss 0.0827
Epoch 10: loss 0.0777


In [24]:
model.eval()
with torch.no_grad():
    outputs = model(x_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print(f"Test Accuracy: {accuracy:.4f}")

torch.save(model.state_dict(), 'email_reply_model.pth')
torch.save(vectorizer, 'vectorizer.pth')

Test Accuracy: 0.9436


In [25]:

def predict_reply(email_body):
    """Predict if an email should be replied to."""
    model.eval()
    X_new = vectorizer.transform([email_body]).toarray()
    X_tensor = torch.tensor(X_new, dtype=torch.float32)
    with torch.no_grad():
        output = model(X_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item() == 1  # True if replied


In [30]:
test_body = '''
    We would like to schedule a call with you.  
'''

predict_reply(test_body)

True